# Implementacja algorytmów Minimax i Alfa-Beta dla gry Clobber

Mikołaj Kubś, 272662

## Opis użycia bibliotek

### Standardowe biblioteki

- abc - definiowanie klas i metod abstrakcyjnych
- dataclasses - do definiowa klas danych, których strukturę można zamrozić i łatwo porównywać
- random - do generowania losowych liczb, przydatne było tylko w jednej, testowej heurystyce (jedyna niedeterministyczna)

### Zewnętrzne biblioteki

- geopy - do obliczania odległości między przystankami
- pandas - do szybszego przetworzenia pliku CSV
- Cython - generowanie kodu C z kodu podobnego do języka Python
- pytest - testy poprawności działania gry Clobber

## Clobber

Clobber to abstrakcyjna gra planszowa dla dwóch graczy, rozgrywana na siatce (np. 8×8). Na początku każdy gracz ma swoje pionki umieszczone na polach w ich kolorze na szachownicy. Gracze na zmianę wykonują ruchy: przesuwają jeden ze swoich pionków na pole sąsiadujące w pionie lub poziomie, zajmując miejsce pionka przeciwnika — "clobberując" go, czyli usuwając z planszy.

Celem gry jest unieruchomienie przeciwnika — gracz przegrywa, jeśli nie może wykonać legalnego ruchu.

## Kod gry Clobber

Kod napisany jest w języku Cython i generowany jest z niego osobny moduł

### Dodatkowe pliki przydatne do poprawnego typowania itd.

```py
# board.pxd

cdef class Board:
    cdef public int n
    cdef public int m
    cdef public int turn
    cdef public list state

    cpdef get_piece_at(self, tuple position)
    cpdef replace_piece_at(self, tuple position, object new_piece)
    cpdef list get_neighbours_positions_filtered(self, tuple position, object piece_filter)
    cpdef list get_all_pieces(self, object piece)
    cpdef bint has_moves(self, bint white)
    
# board.pyi

from typing import Callable, List, Tuple, Dict, Any
from clobber.types import piece_type, move


class Board:
    n: int
    m: int
    turn: int
    state: List[str]

    def __init__(self, n: int, m: int,
                 state: List[str], turn: int = 0) -> None: ...

    @staticmethod
    def initialize_board(
        n: int, m: int) -> 'Board': ...

    def copy(self) -> 'Board': ...

    def __str__(self) -> str: ...
    def pretty(self) -> str: ...

    def get_piece_at(self, position: Tuple[int, int]) -> piece_type: ...

    def replace_piece_at(
        self, position: Tuple[int, int], new_piece: piece_type) -> piece_type: ...

    def get_neighbours_positions(
        self, position: Tuple[int, int]) -> List[Tuple[int, int]]: ...
    def get_neighbours_positions_filtered(self, position: Tuple[int, int], piece_filter: Callable[[
                                          piece_type], bool]) -> List[Tuple[int, int]]: ...

    def make_move(self, new_color: piece_type, m: move) -> None: ...
    def move_assuming_correct(
        self, new_color: piece_type, m: move) -> None: ...

    def get_all_pieces(self, piece: piece_type) -> List[Tuple[int, int]]: ...
    def generate_moves(
        self, for_white: bool) -> Dict[Tuple[int, int], List[Tuple[int, int]]]: ...

    def has_moves(self, white: bool) -> bool: ...

# types.pxd

ctypedef tuple move

# types.py

from typing import Literal

piece_type = Literal['B'] | Literal['W'] | Literal['_'] | Literal['outside']
move = tuple[tuple[int, int], tuple[int, int]]
```

Gra działa, mając reprezentację planszy jako lista stringów ("B" lub "W" lub "_"). Pozwala na kopiowanie, uzyskanie czytelnej postaci, sąsiadów danej pozycji (ze wsparciem funkcyjnym na filtr), poruszania się pionkami czy generację możliwych ruchów.

## Testy Clobber

In [1]:
import pytest

from clobber.board import Board


@pytest.mark.parametrize(("dimensions", "expected"),
                         [
    ((5, 6), "W B W B W B\nB W B W B W\nW B W B W B\nB W B W B W\nW B W B W B"),
    ((10, 10), "W B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W")
])
def test_board_generation(dimensions: tuple[int, int], expected: str) -> None:
    board: Board = Board.initialize_board(*dimensions)

    assert str(board) == expected


def test_neighbours() -> None:
    board: Board = Board.initialize_board(5, 6)

    black_neighbours: list[tuple[int, int]
                           ] = board.get_neighbours_positions_filtered((1, 1), lambda p: p == 'B')
    assert (0, 1) in black_neighbours
    assert (2, 1) in black_neighbours
    assert (1, 2) in black_neighbours
    assert (1, 0) in black_neighbours
    assert len(black_neighbours) == 4
    white_neighbours: list[tuple[int, int]
                           ] = board.get_neighbours_positions_filtered((1, 1), lambda p: p == 'W')
    assert len(white_neighbours) == 0


def test_get_piece() -> None:
    board: Board = Board.initialize_board(5, 6)

    assert board.get_piece_at((5, 4)) == 'B'
    assert board.get_piece_at((5, 5)) == 'outside'
    assert board.get_piece_at((6, 4)) == 'outside'


def test_generate_moves() -> None:
    board: Board = Board.initialize_board(2, 2)

    moves: dict[tuple[int, int], list[tuple[int, int]]
                ] = board.generate_moves(True)

    assert moves[(0, 0)] == [(1, 0), (0, 1)]
    assert moves[(1, 1)] == [(0, 1), (1, 0)]


def test_replace_piece() -> None:
    board: Board = Board.initialize_board(2, 2)

    board.replace_piece_at((0, 0), '_')
    assert board.get_piece_at((0, 0)) == '_'

    board.replace_piece_at((1, 1), '_')
    assert board.get_piece_at((1, 1)) == '_'

    board.replace_piece_at((0, 0), 'B')
    assert board.get_piece_at((0, 0)) == 'B'

    board.replace_piece_at((1, 0), 'B')
    assert board.get_piece_at((1, 0)) == 'B'


def test_move() -> None:
    board: Board = Board.initialize_board(5, 6)

    board.move_assuming_correct('W', ((0, 0), (1, 0)))

    assert board.get_piece_at((0, 0)) == '_'
    assert board.get_piece_at((1, 0)) == 'W'


def test_get_all_pieces() -> None:
    board: Board = Board.initialize_board(3, 3)

    white_positions: list[tuple[int, int]] = board.get_all_pieces('W')
    black_positions: list[tuple[int, int]] = board.get_all_pieces('B')

    assert len(white_positions) == 5
    assert len(black_positions) == 4
    assert (0, 0) in white_positions
    assert (1, 0) in black_positions


def test_has_moves() -> None:
    board: Board = Board.initialize_board(2, 2)

    assert board.has_moves(True)
    assert board.has_moves(False)

    board.replace_piece_at((0, 0), '_')
    board.replace_piece_at((1, 0), '_')
    board.replace_piece_at((0, 1), '_')
    board.replace_piece_at((1, 1), '_')

    assert not board.has_moves(True)
    assert not board.has_moves(False)


def test_copy() -> None:
    board: Board = Board.initialize_board(2, 2)
    board_copy: Board = board.copy()

    assert str(board) == str(board_copy)

    board_copy.replace_piece_at((0, 0), '_')
    assert board.get_piece_at((0, 0)) == 'W'
    assert board_copy.get_piece_at((0, 0)) == '_'


def test_turn() -> None:
    board: Board = Board.initialize_board(5, 5)

    assert board.turn == 0
    board.move_assuming_correct('W', ((0, 0), (0, 1)))
    assert board.turn == 1
    new_board: Board = board.copy()
    assert new_board.turn == 1
    board.move_assuming_correct('W', ((0, 1), (1, 0)))
    assert new_board.turn == 1
    assert board.turn == 2
